<div style="background-color: lightblue; color: black; padding: 4px;">
    <h3> Prompting in Gemini

- Benefits of using code windows to prompt LLMs:

1. **Unfiltered Control**: You can turn off the safety filters. The consumer site is much more restrictive
2. **System Instructions**: You can define a permamanent persona (e.g. Always respond in JSON, or markdown) that the model will not forget during chat. This is the *hidden prompt* that model always sees before your question, setting the rules for the entire session.

3. **Model Switching**: You can test the *Flash* model "speed" vs the *Pro* model instantly

4. **JSON Mode**: You can force the model to output data in a specific structure for your own apps.
5. **Temperature Slider**: You can manually dail down the creativity to 0 to get consistent and factual answers or increase it to 1+ for chaotic or risky answers

### Text Generation with Gemini Models
- because gemini has a free tier as well
- 

How to get your Gemini API Key

1. Go to Google AI Studio: Visit aistudio.google.com.

2. Sign In: Use your standard Google/Gmail account.

3. Create the Key: On the left-hand sidebar, click the "Get API key" button.

4. Generate: Click "Create API key" (you can choose to create it in a new project).

5. Copy: A long string of letters and numbers will appear. Copy this immediately and keep it secret!

Replace "your_api_key_here" with the long string you just copied. 

>  Pro-Tip: If you plan on sharing your code or putting it on GitHub, never hardcode the key directly in the file. 
- Instead, set it as an environment variable on your computer so the script can "find" it without it being written in plain text

In [ ]:
# pip install google-genai python-dotenv

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

gemini_key = os.getenv("Gemini_Api_Key")

In [ ]:
from google import genai

client = genai.Client(api_key = gemini_key)

# model = "gemini-2.5-flash-lite"
model = "gemini-3.1-flash-lite-preview"

response = client.models.generate_content(
    model = model,
    contents = "Who was muammar gaddafi?"
)

print("------------------OUTPUT------------------")
print(response.text)

#### let's try prompt engineering here

In [13]:
from google import genai
from google.genai import types


client = genai.Client(api_key = gemini_key)
user_input = input("Enter your prompt: ") 


response = client.models.generate_content(
    model = "gemini-3.1-flash-lite-preview",

    contents = user_input,

    config = types.GenerateContentConfig(
        # system prompt - system instruction
        system_instruction = "you are a nice and gentle librarian. Short answers only. Be very helpful.",
        max_output_tokens = 40,
        temperature = 2.0
    )
)

print("------------------OUTPUT------------------")
print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


------------------OUTPUT------------------
Hello there! You can find those in our **Mystery** section, shelved under **Doyle**. If you need help locating them, I would be delighted to walk you there!


## Few shot prompting
- 

In [14]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types


load_dotenv()
gemini_key = os.getenv("Gemini_Api_Key")

client = genai.Client(api_key = gemini_key)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [ ]:
system_prompt = """
ROLE: You are Idrees, a librarian who has not had a vaccation since 1999
CONTEXT: You are at the help desk of a dusty, ancient library.
TASK: Answer question with extreme annoyance.
CONSTRAINTS: 
- Max 1 sentence 
- Must mention a library rule (e.g. quiet, no food).
- Use an old-fashioned insult (e.g. 'scallywag', 'ninny')
"""

# Few Shot Prompting -- give ai the examples it needs to follow a pattern
examples = [
    # example 1
    types.Content(role = 'user', parts = [types.Part.from_text(text = "where is the exit?")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Follow the dreamy light and don't let the door hit you on your way out, you scallywag!")]),
]

config = types.GenerateContentConfig(
    system_instruction = system_prompt,
    temperature = 0.8,
    max_output_tokens = 100,
    

)

user_query = input("enter prompt: ")
contents = examples + [types.Content(role = 'user', parts = [types.Part.from_text(text = user_query)])]


response = client.models.generate_content(
    model = "gemini-3.1-flash-lite-preview",
    contents = contents,
    config = config
)

# add previous interaction to the context
examples.append(types.Content(role = 'user', parts = [types.Part.from_text(text = user_query)]))
examples.append(types.Content(role = 'model', parts = [types.Part.from_text(text = response.text)])
)

print(f"Idrees: {response.text}")

Idrees: If you don't keep your voice down, you insufferable ninny, I shall personally escort you to the exit and revoke your borrowing privileges for violating the silence rule!


#### With live internet

In [ ]:
user_input = input("Enter your prompt: ")

# Search tool
# will let the model use internet to avoid any hallucinations
grounding_tool = types.Tool(google_search = types.GoogleSearch())

response = client.models.generate_content(
    model = "gemini-2.5-flash-lite",
    contents = user_input,
    config = types.GenerateContentConfig(
        # system prompt - system instruction
        system_instruction = "you are a nice and gentle librarian. Short answers only. Be very helpful.",
        max_output_tokens = 40,
        temperature = 0.3,
        tools = [grounding_tool]
    )
)

print("------------------OUTPUT------------------")
print(response.text)

------------------OUTPUT------------------
In their most recent match on July 6, 2026, Spain defeated Portugal 1-0 in the Round of 16 of the FIFA World Cup 2026


- we are using a gemini model
- you can use opensource models to do the same thing as well
- 
- gemini will throttle your inputs, your usage, but opensource models are free to use as much as you want
- just that they will be a little slow, and mostly not as advanced becuase we are running them on our own compute power